# RouteProfile — Richer Model Embeddings for LLMRouter

This notebook demonstrates the full **RouteProfile pipeline**, which generates higher-quality LLM model embeddings by exploiting the rich graph structure among models, architectures, benchmark datasets, and query domains.

These embeddings drop in as replacements for the default Longformer text encodings used by **GraphRouter**, **MFRouter**, and **PersonalizedRouter**.

## Pipeline overview

```
JSON data files
      │
      ▼
 Build HeteroGraph (.pt)          ← Step 1
      │
      ▼
 Generate model profile (.npz)    ← Step 2  (choose one method)
      │
      ▼
 Apply to llm_data.json           ← Step 3  (inject embeddings)
      │
      ▼
 Train router as usual            ← Step 4
```

## Profile methods at a glance

| Method | How | Training | GPU |
|--------|-----|----------|-----|
| `flat` | average Longformer embeddings of sampled text | no | no |
| `emb_gnn` | K-hop graph propagation of Longformer embeddings | no | no |
| `index` | random orthogonal baseline | no | no |
| `trainable_gnn` | self-supervised HANConv on graph | yes | optional |
| `text_gnn` | local LLM (vLLM) + graph propagation | no | yes |


## 1. Environment Setup

In [ ]:
# Install required packages
# !pip install -e '../../.[routeprofile]'

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path

# Verify routeprofile is available
from llmrouter.routeprofile.data import get_profile_data_dir, get_bundled_path
print(f"Profile data directory: {get_profile_data_dir()}")
print(f"Bundled JSON files found: {os.listdir(get_profile_data_dir())}")

## 2. Step 1 — Build the Heterogeneous Graph

RouteProfile encodes LLM metadata as a **heterogeneous PyTorch Geometric graph** with the following node types:

- **model** — each LLM candidate (Longformer-encoded description + benchmark scores as edge weights)
- **architecture** — model family / architecture (e.g., `transformer-decoder`)
- **dataset** — benchmark task (e.g., `MMLU`, `HumanEval`)
- **domain** — high-level capability domain (e.g., `coding`, `reasoning`)
- **query** — representative user queries per task

Five graph types are available, from minimal to richest:

| Graph type | Node types |
|------------|------------|
| `task` | model, architecture, dataset |
| `query` | model, query |
| `query_task` | model, architecture, dataset, query |
| `task_domain` | model, architecture, dataset, domain |
| `query_task_domain` | all 5 |

We'll use `task_domain` (a good balance of richness and speed).

In [ ]:
import tempfile

# Use a temp directory for all outputs in this tutorial
work_dir = Path(tempfile.mkdtemp(prefix="routeprofile_"))
graph_dir = work_dir / "graphs"
profile_dir = work_dir / "profiles"
graph_dir.mkdir()
profile_dir.mkdir()

print(f"Working directory: {work_dir}")

In [ ]:
from llmrouter.routeprofile.build_data_graph.build_task_domain_graph import build_task_domain_graph

graph_path = str(graph_dir / "task_domain_graph.pt")

print("Building task_domain graph (may take ~1-2 min for Longformer encoding)...")
build_task_domain_graph(mode="standard", save=graph_path)
print(f"Graph saved to: {graph_path}")

In [ ]:
# Inspect the graph
graph = torch.load(graph_path, weights_only=False)
print("Node types:", graph.node_types)
print("Edge types:", graph.edge_types)
print()
for nt in graph.node_types:
    x = graph[nt].x
    names = getattr(graph[nt], 'node_names', None)
    n_names = len(names) if names is not None else '—'
    print(f"  {nt}: {x.shape[0]} nodes × {x.shape[1]} dims  (node_names: {n_names})")

## 3. Step 2 — Generate Model Profile Embeddings

Given the graph, we now run one of the profile methods to produce a `.npz` file mapping `{model_name → np.ndarray[768]}`.

### 3a. emb_gnn (recommended, training-free)

`emb_gnn` runs K-hop graph propagation over the Longformer node features — no training required, deterministic, fast.

In [ ]:
from llmrouter.routeprofile.get_model_profile.training_free.emb_gnn_profile import build_model_profile

emb_gnn_path = str(profile_dir / "emb_gnn.npz")

build_model_profile(
    graph=graph_path,
    K=2,           # number of propagation hops
    norm="sym",    # symmetric normalisation (best for most graphs)
    normalize=True, # L2-normalise the final embeddings
    save=emb_gnn_path,
)
print(f"emb_gnn profile saved: {emb_gnn_path}")

In [ ]:
# Inspect the profile
profile = np.load(emb_gnn_path)
print(f"Models in profile: {len(profile.files)}")
print(f"First 5 models: {profile.files[:5]}")
print(f"Embedding shape: {profile[profile.files[0]].shape}")
print(f"Embedding dtype: {profile[profile.files[0]].dtype}")

# Check L2 norms (should be ~1.0 since normalize=True)
norms = [np.linalg.norm(profile[m]) for m in profile.files]
print(f"L2 norms: min={min(norms):.4f}, max={max(norms):.4f} (expected ~1.0)")

### 3b. flat (simple text baseline, no graph structure)

In [ ]:
from llmrouter.routeprofile.get_model_profile.training_free.flat_profile import build_model_profile as build_flat

flat_path = str(profile_dir / "flat.npz")
build_flat(graph=graph_path, seed=42, save=flat_path)
print(f"flat profile saved: {flat_path}")

### 3c. trainable_gnn (best quality, requires short training)

In [ ]:
from llmrouter.routeprofile.get_model_profile.trainable.trainable_gnn_profile import build_model_profile as build_trainable

trainable_emb_path  = str(profile_dir / "trainable_gnn.npz")
trainable_ckpt_path = str(profile_dir / "trainable_gnn_ckpt.pt")

build_trainable(
    graph=graph_path,
    save_emb=trainable_emb_path,
    save_ckpt=trainable_ckpt_path,
    epochs=50,   # increase to 200+ for best results
    seed=42,
)
print(f"trainable_gnn profile saved: {trainable_emb_path}")
print(f"Checkpoint saved: {trainable_ckpt_path}")

### 3d. text_gnn (highest quality, requires vLLM + GPU)

`text_gnn` replaces numeric graph propagation with **LLM-generated text summaries** at each hop.  
Each model node's description is rewritten by a local LLM (via vLLM) to incorporate its neighbours' texts and benchmark scores. The final summary is then encoded by Longformer into a 768-dim embedding.

**Requirements:** `pip install 'llmrouter-lib[routeprofile-text-gnn]'`

**K=0 special case:** skips the LLM call entirely and just Longformer-encodes the raw `node_feature_text` — useful as a CPU baseline with no vLLM needed.

In [ ]:
import importlib

if importlib.util.find_spec("vllm") is not None:
    from llmrouter.routeprofile.get_model_profile.training_free.text_gnn_profile import build_model_profile as build_text_gnn

    text_gnn_path = str(profile_dir / "text_gnn.npz")
    build_text_gnn(
        graph=graph_path,
        K=2,                                      # number of LLM aggregation hops
        model="Qwen/Qwen2.5-7B-Instruct",         # any vLLM-compatible model
        tp=1,                                     # number of GPUs (tensor parallel)
        gpu_memory_utilization=0.9,
        emb_save=text_gnn_path,
        text_save=str(profile_dir / "text_gnn_texts.json"),
    )
    print(f"text_gnn profile saved: {text_gnn_path}")
else:
    print("vllm not installed — skipping text_gnn.")
    print("Install with: pip install 'llmrouter-lib[routeprofile-text-gnn]'")
    text_gnn_path = None

# K=0 baseline: no LLM call, just Longformer on raw node_feature_text (CPU-only)
# build_text_gnn(graph=graph_path, K=0, emb_save=str(profile_dir / "text_gnn_k0.npz"))

### Comparing methods

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

methods = {
    "emb_gnn": np.load(emb_gnn_path),
    "flat": np.load(flat_path),
    "trainable_gnn": np.load(trainable_emb_path),
}

# Use common models across all methods
common_models = list(set.intersection(*[set(m.files) for m in methods.values()]))
print(f"Common models across all methods: {len(common_models)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (method_name, profile) in zip(axes, methods.items()):
    X = np.stack([profile[m] for m in common_models])
    pca = PCA(n_components=2)
    X2d = pca.fit_transform(X)
    ax.scatter(X2d[:, 0], X2d[:, 1], alpha=0.7)
    for i, name in enumerate(common_models):
        ax.annotate(name[:12], (X2d[i, 0], X2d[i, 1]), fontsize=6)
    ax.set_title(f"{method_name}\nPCA of 768-dim embeddings")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.savefig(str(work_dir / "embedding_comparison.png"), dpi=120)
plt.show()
print("PCA plot saved.")

## 4. Step 3 — Inject Embeddings into llm_data.json

**GraphRouter** and **MFRouter** read model embeddings from the `"embedding"` field of `llm_data.json`.  
We use `npz_to_llm_embeddings_json` to merge the RouteProfile output into that file.

This leaves all other metadata (API endpoints, size, features, etc.) intact.

In [ ]:
from llmrouter.routeprofile.utils import npz_to_llm_embeddings_json

# Path to your LLMRouter llm_data JSON
llm_data_path = str(PROJECT_ROOT / "data" / "example_data" / "llm_candidates" / "default_llm.json")

# Output: new file with RouteProfile embeddings
output_path = str(work_dir / "default_llm_routeprofile.json")

npz_to_llm_embeddings_json(
    npz_path=emb_gnn_path,
    llm_data_path=llm_data_path,
    output_path=output_path,
    missing="warn",   # 'warn' | 'skip' | 'error'
)

print(f"Updated llm_data saved to: {output_path}")

In [ ]:
# Verify the output
with open(output_path) as f:
    result = json.load(f)

sample_model = next(iter(result))
print(f"Sample model: {sample_model}")
print(f"Fields: {list(result[sample_model].keys())}")
emb = result[sample_model].get("embedding")
if emb:
    print(f"Embedding dim: {len(emb)}")
    print(f"Embedding preview: {emb[:5]}...")
else:
    print("No embedding — model not found in RouteProfile output (check missing= mode)")

## 5. Step 3 (alt) — Convert to .pkl for PersonalizedRouter

**PersonalizedRouter** loads embeddings from a `.pkl` file (`llm_embedding_data` in its YAML config).  
Use `npz_to_pkl` to generate the right format.

In [ ]:
import pickle
from llmrouter.routeprofile.utils import npz_to_pkl

pkl_path = str(work_dir / "model_embeddings.pkl")
npz_to_pkl(emb_gnn_path, pkl_path)

print(f".pkl saved to: {pkl_path}")

with open(pkl_path, "rb") as f:
    pkl_data = pickle.load(f)

print(f"Models in .pkl: {len(pkl_data)}")
sample = next(iter(pkl_data))
print(f"Sample model: {sample}")
print(f"Type: {type(pkl_data[sample])}, shape: {pkl_data[sample].shape}")

## 6. Step 4 — Use New Embeddings in Router Training

Update your router YAML to point to the new embedding file. **No router code changes needed.**

### GraphRouter / MFRouter (llm_data with embedded embeddings)

```yaml
# configs/model_config_train/graphrouter.yaml
data_path:
  llm_data: '/path/to/default_llm_routeprofile.json'  # ← use the output from Step 3
  train_data: 'data/example_data/...'
  test_data:  'data/example_data/...'
```

### PersonalizedRouter (separate .pkl file)

```yaml
# configs/model_config_train/personalizedrouter.yaml
data_path:
  llm_embedding_data: '/path/to/model_embeddings.pkl'  # ← use the output from Step 3 (alt)
  train_data: 'data/example_data/...'
  test_data:  'data/example_data/...'
```

Then train normally:

```bash
llmrouter train --router graphrouter --config configs/model_config_train/graphrouter.yaml
```

## 7. CLI Quick Reference

All three pipeline steps are also available as CLI commands:

```bash
# Step 1: Build graph
llmrouter profile build-graph \
    --graph-type task_domain \
    --mode standard \
    --output-dir /tmp/graphs/

# Step 2: Generate profile
llmrouter profile build-profile \
    --method emb_gnn \
    --graph /tmp/graphs/task_domain_graph_full.pt \
    --output /tmp/profiles/emb_gnn.npz \
    --K 2 --normalize

# Step 3: Apply to llm_data.json
llmrouter profile apply \
    --profile /tmp/profiles/emb_gnn.npz \
    --llm-data data/example_data/llm_candidates/default_llm.json \
    --output data/example_data/llm_candidates/default_llm_routeprofile.json
```

Get help for any subcommand:

```bash
llmrouter profile --help
llmrouter profile build-graph --help
llmrouter profile build-profile --help
llmrouter profile apply --help
```

## 8. Using Custom Data

By default, RouteProfile uses the **bundled JSON data files** in `llmrouter/routeprofile/data/profile_data/`. You can override these with your own data:

```python
from llmrouter.routeprofile.build_data_graph.build_task_domain_graph import build_task_domain_graph

# Pass custom paths to override bundled defaults
build_task_domain_graph(
    mode="newllm",  # 'standard' or 'newllm'
    model_feature_path="/path/to/my_model_feature.json",
    save="/path/to/my_graph.pt",
)
```

**model_feature.json format:**
```json
{
  "my-model-name": {
    "feature": "A text description of the model...",
    "architecture": "transformer-decoder",
    "detailed_scores": {
      "MMLU": 0.82,
      "HumanEval": 0.45
    }
  }
}
```

## Summary

You've seen the full RouteProfile pipeline:

1. **Build graph** — encode LLM metadata as a heterogeneous PyG graph
2. **Generate profile** — run a profile method (emb_gnn, flat, trainable_gnn, …) to produce 768-dim embeddings per model
3. **Apply** — inject embeddings into `llm_data.json` (or `.pkl`) without changing any router code
4. **Train** — point the router YAML to the new embedding file and train normally

The graph-enriched embeddings capture **relationships between models, architectures, benchmarks, and domains** that simple per-model text encoding misses, leading to better router performance when training data is limited.

See `llmrouter/routeprofile/README.md` for the full API reference.